# Lab 2: ice, water and vapour — 2025 version

This is last year's simulation and its original analysis questions, adapted to
Atomify's current project/run folders. The separate heater pilot is not included.

1. Choose **Use as project** so you can edit the input between stages.
2. Run the supplied first stage and wait for it to finish.
3. Run the notebook below. Its final cell copies the saved configuration into
   the project's files, ready for the next stage.
4. Change `Told`, `Tnew`, `runlength` and the ensemble/expansion lines as directed
   by the lab instructions, then run again. Re-run the notebook after each stage.

The input initially uses `Told = 250`, `Tnew = 251` and `runlength = 1000`.
The notebook selects the latest run log automatically. A specific older log can
be selected by replacing `logfile` with its path.

**Legacy numerical limitations:** the original short runs and expansion settings
are retained for reviewing the exercise workflow. The expanding PPPM grid can
lose accuracy, and the dilute stage can be slow in a browser. These traces do
not establish equilibrium latent heats. The starting configuration and LAMMPS
input are unchanged from Dag's example.


In [ ]:
%pip install lammps-logfile pandas
%matplotlib inline

Open logfile and display which data are available and what the keywords for accessing this data are.

In [ ]:
from pathlib import Path
import lammps_logfile
import matplotlib

logs = sorted(Path("runs").glob("*/log_*K.lammps"))
if not logs:
    raise FileNotFoundError("Finish a simulation first, then run this notebook again.")
logfile = logs[-1]
print("Reading:", logfile)
log = lammps_logfile.File(str(logfile))
print("Log keywords: ", log.get_keywords())

Choose and import data from logfile and plot data.

Study the evolution of total energy and enthalpy to consider 
1. if the system has reached equilibrium
2. where the energy goes

In [ ]:
s = log.get("Step")
T = log.get("Temp")
E = log.get("TotEng")
H = log.get("Enthalpy")

import matplotlib.pyplot as plt
plt.plot(s, E)
plt.xlabel('Step number')
plt.ylabel('Total energy')
plt.title('Total energy')
plt.show()
plt.plot(s, H)
plt.xlabel('Step number')
plt.ylabel('Enthalpy')
plt.title('Enthalpy')
plt.show()
plt.plot(H, T)
plt.xlabel('Enthalpy')
plt.ylabel('Temperature')
plt.title('Temperature vs Enthalpy')
plt.show()


## Keep the configuration for the next stage

Atomify stores each run in its own folder. The next simulation needs its starting
`data.<temperature>K` file among the project files. After the run has finished,
this cell copies that checkpoint back to the project directory, replacing an
older checkpoint of the same name if present. Positions, velocities and molecular
topology are preserved; no new system is generated.

Then edit the input so `Told` matches the saved temperature. For the expansion
stage, follow the lab's changes to the thermostat and `fix deform` lines too.


In [ ]:
import shutil

checkpoint_name = "data." + logfile.stem.removeprefix("log_")
checkpoint = logfile.parent / checkpoint_name
if not checkpoint.is_file():
    raise FileNotFoundError(
        f"No {checkpoint_name} in {logfile.parent}. Wait for the run to finish; "
        "if it failed, inspect the simulation console before continuing."
    )
shutil.copyfile(checkpoint, Path(checkpoint_name))
print(f"Ready for the next stage: {checkpoint_name}")